# EBA-Style Stress Test — Greek Systemic Banks

Applies an adverse macro scenario to baseline 2024 financials and estimates impact on net profit and CET1.

## Adverse Scenario Parameters

| Shock | Adverse | Source / Rationale |
|-------|---------|--------------------|
| Cost of Risk | +200bps vs 2024 baseline | EBA 2023 stress test adverse scenario benchmark |
| Loan Volume | −15% vs 2024 | Credit tightening under adverse macro |
| NIM Compression | −50bps vs 2024 | Spread compression, deposit repricing lag |
| Operating Costs | Flat (no cost cutting) | Conservative assumption |
| Tax Rate | Unchanged | Constant effective rate |

**Methodology:**
1. Adverse NII = (2024 NIM − 50bps) × (2024 Loans × 85%)
2. Adverse Impairment = −(2024 Loans × 85% × (2024 CoR + 200bps))
3. Adverse PPOP = Adverse NII + Other Income (fixed at 2024) − 2024 OpEx
4. Adverse PBT = Adverse PPOP + Adverse Impairment
5. Adverse Net Profit = Adverse PBT × (1 − effective tax rate)
6. Adverse CET1 = 2024 CET1 − (Adverse net profit impact / 2024 RWA proxy)
   - RWA proxy: Loans × 0.75 (50% average risk weight on loan book)

> **Disclaimer**: Simplified 1-year static balance sheet stress. Does not model sovereign bond haircuts,
> funding cost shocks, or dynamic balance sheet responses. For illustration only.

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from pathlib import Path

COLORS = {
    'Eurobank':    '#0067B1',
    'Alpha Bank':  '#E2231A',
    'Piraeus Bank':'#F7A600',
    'NBG':         '#003087',
}
BANKS = list(COLORS.keys())
DATA_DIR = Path('../02_Banking_Sector_Dashboard/data/processed')

kpis = pd.read_csv(DATA_DIR / 'kpis_final.csv')
is_long = pd.read_csv(DATA_DIR / 'income_statement_final.csv')

# Get PBT for effective tax rate
is_piv = is_long.pivot_table(index=['bank','year'], columns='metric', values='value', aggfunc='first').reset_index()
df24 = kpis[kpis['year'] == 2024].merge(
    is_piv[is_piv['year']==2024][['bank','Profit before tax']].rename(columns={'Profit before tax':'pbt'}),
    on='bank'
).set_index('bank')

df24['eff_tax_rate'] = 1 - df24['net_profit'] / df24['pbt']
df24['cost_of_risk_2024'] = df24['impairment'].abs() / df24['loans']  # as decimal
df24['nim_dec_2024']      = df24['nim'] / 100  # NIM as decimal

# Estimate other income (operating_income - nii - fee income not in kpis → use residual)
# Other income proxy = operating_income - nii (fees bundled with other for simplicity)
df24['other_income_proxy'] = df24['operating_income'] - df24['nii']

print('2024 Baseline:')
print(df24[['net_profit','roe','cet1','cost_of_risk_2024','nim','loans']].round(3).to_string())

In [ ]:
# ── Stress scenario parameters ────────────────────────────────────────────────
STRESS = {
    'cor_shock_bp':   200,   # +200bps additional cost of risk
    'loan_shock_pct': -0.15, # -15% loan volume
    'nim_shock_bp':   -50,   # -50bps NIM compression
}

# RWA approximation: risk weight loan book at 50% avg, 
# then RWA = loans × 0.50 (for CET1 impact calculation)
RWA_WEIGHT = 0.50

results = []
for bank in BANKS:
    row = df24.loc[bank]

    # Adverse values
    loans_adv  = row['loans']   * (1 + STRESS['loan_shock_pct'])
    nim_adv    = row['nim_dec_2024'] + STRESS['nim_shock_bp'] / 10000
    cor_adv    = row['cost_of_risk_2024'] + STRESS['cor_shock_bp'] / 10000

    nii_adv        = nim_adv * row['total_assets'] * (1 + STRESS['loan_shock_pct'])  # assets shrink with loans
    impairment_adv = -(cor_adv * loans_adv)  # negative (provision charge)
    opex_adv       = row['operating_expenses']  # flat cost base
    other_inc_adv  = row['other_income_proxy']  # fixed

    opincome_adv = nii_adv + other_inc_adv
    ppop_adv     = opincome_adv + opex_adv   # opex is negative
    pbt_adv      = ppop_adv + impairment_adv
    net_profit_adv = pbt_adv * (1 - row['eff_tax_rate'])

    # Equity impact: adverse net profit vs baseline
    equity_delta = net_profit_adv - row['net_profit']
    equity_adv   = row['equity'] + equity_delta

    # CET1 impact (simplified): (equity_delta / RWA)
    rwa_proxy = row['loans'] * RWA_WEIGHT  # baseline RWA proxy
    cet1_delta_pp = (equity_delta / rwa_proxy) * 100
    cet1_adv = row['cet1'] + cet1_delta_pp

    roe_adv = net_profit_adv / equity_adv * 100

    results.append({
        'bank': bank,
        # Baseline
        'nii_base':        row['nii'],
        'net_profit_base': row['net_profit'],
        'roe_base':        row['roe'],
        'cet1_base':       row['cet1'],
        # Adverse
        'nii_adv':        nii_adv,
        'net_profit_adv': net_profit_adv,
        'roe_adv':        roe_adv,
        'cet1_adv':       cet1_adv,
        # Deltas
        'delta_net_profit': net_profit_adv - row['net_profit'],
        'delta_cet1_pp':    cet1_delta_pp,
    })

st = pd.DataFrame(results).set_index('bank')
print('Stress Test Results:')
print(st[['net_profit_base','net_profit_adv','delta_net_profit','roe_base','roe_adv','cet1_base','cet1_adv','delta_cet1_pp']].round(1).to_string())

In [ ]:
# ── Chart 1: Baseline vs Adverse — Net Profit & CET1 ─────────────────────────
fig1 = make_subplots(
    rows=1, cols=2,
    subplot_titles=['<b>Net Profit: Baseline vs Adverse (€m)</b>', '<b>CET1: Baseline vs Adverse (%)</b>'],
    horizontal_spacing=0.12,
)

x = BANKS
# Net profit
fig1.add_trace(go.Bar(
    x=x, y=[st.loc[b,'net_profit_base'] for b in BANKS],
    name='Baseline 2024', marker_color=[COLORS[b] for b in BANKS], opacity=0.9,
    text=[f'{st.loc[b,"net_profit_base"]:.0f}m' for b in BANKS], textposition='outside',
), row=1, col=1)
fig1.add_trace(go.Bar(
    x=x, y=[st.loc[b,'net_profit_adv'] for b in BANKS],
    name='Adverse scenario', marker_color='#666666', opacity=0.85,
    text=[f'{st.loc[b,"net_profit_adv"]:.0f}m' for b in BANKS], textposition='outside',
), row=1, col=1)
fig1.add_hline(y=0, line_color='rgba(255,255,255,0.4)', row=1, col=1)

# CET1
fig1.add_trace(go.Bar(
    x=x, y=[st.loc[b,'cet1_base'] for b in BANKS],
    name='Baseline CET1', marker_color=[COLORS[b] for b in BANKS], opacity=0.9,
    text=[f'{st.loc[b,"cet1_base"]:.1f}%' for b in BANKS], textposition='outside',
    showlegend=False,
), row=1, col=2)
fig1.add_trace(go.Bar(
    x=x, y=[st.loc[b,'cet1_adv'] for b in BANKS],
    name='Adverse CET1', marker_color='#666666', opacity=0.85,
    text=[f'{st.loc[b,"cet1_adv"]:.1f}%' for b in BANKS], textposition='outside',
    showlegend=False,
), row=1, col=2)
fig1.add_hline(y=10.5, line_dash='dash', line_color='#EF553B',
               annotation_text='ECB min CET1 10.5%', annotation_position='bottom right',
               row=1, col=2)

fig1.update_layout(
    title_text=f'<b>EBA-Style Stress Test Results</b> | CoR +{STRESS["cor_shock_bp"]}bps | Loans {STRESS["loan_shock_pct"]*100:.0f}% | NIM {STRESS["nim_shock_bp"]}bps',
    title_font_size=15, template='plotly_dark', height=440, barmode='group',
    legend=dict(orientation='h', y=-0.18, x=0.5, xanchor='center'),
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
)
fig1.show()

In [ ]:
# ── Chart 2: Resilience Ranking ───────────────────────────────────────────────
# Which bank is most resilient? Rank by CET1 remaining above minimum after stress.

ecb_min_cet1 = 10.5  # ECB Pillar 1 + conservation buffer
st['cet1_buffer_adv'] = st['cet1_adv'] - ecb_min_cet1  # headroom above minimum

st_sorted = st.sort_values('cet1_buffer_adv', ascending=False)

fig2 = go.Figure()
fig2.add_trace(go.Bar(
    x=st_sorted.index,
    y=st_sorted['cet1_buffer_adv'],
    marker_color=[COLORS[b] for b in st_sorted.index],
    text=[f'+{v:.1f}pp buffer' for v in st_sorted['cet1_buffer_adv']],
    textposition='outside',
))
fig2.add_hline(y=0, line_color='#EF553B', line_dash='dash',
               annotation_text='ECB minimum (10.5% CET1)', annotation_position='bottom right')

fig2.update_layout(
    title_text='<b>Stress Test Resilience</b> | CET1 headroom above ECB minimum after adverse scenario',
    yaxis_title='CET1 buffer above minimum (pp)',
    template='plotly_dark', height=380,
    paper_bgcolor='#0f1117', plot_bgcolor='#0f1117',
    showlegend=False,
)
fig2.show()

print('\n── Stress Test Resilience Ranking (2024 baseline → adverse) ──')
for rank, (bank, row_s) in enumerate(st_sorted.iterrows(), 1):
    flag = '✅' if row_s['cet1_adv'] > ecb_min_cet1 else '⚠️ BREACH'
    print(f'  #{rank} {bank}: CET1 {row_s["cet1_base"]:.1f}% → {row_s["cet1_adv"]:.1f}% '
          f'(Δ {row_s["delta_cet1_pp"]:+.1f}pp) | Buffer: {row_s["cet1_buffer_adv"]:+.1f}pp {flag}')

In [ ]:
# ── Final validation ──────────────────────────────────────────────────────────
assert len(st) == 4, 'Expected 4 banks'

# All banks should have lower net profit under adverse
assert (st['net_profit_adv'] < st['net_profit_base']).all(), 'Adverse net profit should be below baseline'

# All banks should have lower CET1 under adverse
assert (st['cet1_adv'] < st['cet1_base']).all(), 'Adverse CET1 should be below baseline'

# All banks should remain above ECB minimum under this scenario (given strong CET1 buffers)
# (If any bank breaches, flag it — don't assert, as that's the point of the test)
breaches = st[st['cet1_adv'] < ecb_min_cet1]
if len(breaches) > 0:
    print(f'⚠️  {len(breaches)} bank(s) breach CET1 minimum under adverse scenario:')
    print(breaches[['cet1_base','cet1_adv']].to_string())
else:
    print(f'All {len(st)} banks remain above ECB CET1 minimum ({ecb_min_cet1}%) under adverse scenario.')

print('\n✅ All checks passed.')
most_resilient = st['cet1_buffer_adv'].idxmax()
least_resilient = st['cet1_buffer_adv'].idxmin()
print(f'   Most resilient (largest CET1 buffer after stress): {most_resilient} ({st.loc[most_resilient,"cet1_adv"]:.1f}% CET1)')
print(f'   Most at risk  (smallest CET1 buffer after stress): {least_resilient} ({st.loc[least_resilient,"cet1_adv"]:.1f}% CET1)')
print(f'   Scenario: CoR +{STRESS["cor_shock_bp"]}bps | Loans {STRESS["loan_shock_pct"]*100:.0f}% | NIM {STRESS["nim_shock_bp"]}bps')